In [1]:
import numpy as np
from schemas import (
    ExistingGenerator,
    CandidateGenerator,
    SystemParameters,
    ExpansionInput
)
from CE_model import build_and_solve_expansion

def run_test():
    print("🚀 Προετοιμασία δεδομένων βασισμένων στον αρχικό κώδικα...")

    # 1. Δημιουργία χρονοσειράς ζήτησης (24 ώρες * 14 ημέρες = 336 ώρες)
    hours = 336
    t = np.arange(hours)
    base_demand = 1800 + 700 * np.sin(2 * np.pi * t / 24) + np.random.normal(0, 30, hours)
    demand_profile = np.maximum(base_demand, 400).tolist()

    # Capacity factors
    solar_cf = np.maximum(0, np.sin(2 * np.pi * (t - 6) / 24)).tolist()
    wind_cf = np.clip(0.40 + 0.20 * np.sin(2 * np.pi * t / 80) + np.random.normal(0, 0.05, hours), 0, 1).tolist()

    # 2. Υπάρχον Fleet (GeneratorFleet.csv)
    existing_fleet = [
        ExistingGenerator(
            name="CC1", fuel_type="Natural Gas", capacity_mw=400.0,
            heat_rate=7.2, fuel_cost=4.5, vom_cost=3.0, co2_tons_per_mwh=0.35
        ),
        ExistingGenerator(
            name="CC2", fuel_type="Natural Gas", capacity_mw=300.0,
            heat_rate=7.5, fuel_cost=4.5, vom_cost=3.5, co2_tons_per_mwh=0.40
        ),
        ExistingGenerator(
            name="CT1", fuel_type="Natural Gas", capacity_mw=150.0,
            heat_rate=10.0, fuel_cost=4.5, vom_cost=5.0, co2_tons_per_mwh=0.75
        ),
        ExistingGenerator(
            name="Nuc1", fuel_type="Nuclear", capacity_mw=600.0,
            heat_rate=10.4, fuel_cost=0.8, vom_cost=2.0, co2_tons_per_mwh=0.0
        ),
        ExistingGenerator(
            name="Sol1", fuel_type="Solar", capacity_mw=250.0,
            heat_rate=0.0, fuel_cost=0.0, vom_cost=1.0, co2_tons_per_mwh=0.0,
            is_variable=True
        )
    ]

    # 3. Νέες Υποψήφιες Μονάδες (NewGenerators.csv) - Ακέραια blocks (vb)
    candidate_fleet = [
        CandidateGenerator(
            name="WindNew", fuel_type="Wind", unit_capacity_mw=100.0,
            annual_capex_per_mw=65000.0, op_cost_per_mwh=2.0, co2_tons_per_mwh=0.0,
            is_variable=True, is_integer=True
        ),
        CandidateGenerator(
            name="CCNew", fuel_type="Natural Gas", unit_capacity_mw=250.0,
            annual_capex_per_mw=85000.0, op_cost_per_mwh=35.0, co2_tons_per_mwh=0.30,
            is_variable=False, is_integer=True
        ),
        CandidateGenerator(
            name="CTNew", fuel_type="Natural Gas", unit_capacity_mw=100.0,
            annual_capex_per_mw=45000.0, op_cost_per_mwh=65.0, co2_tons_per_mwh=0.60,
            is_variable=False, is_integer=True
        )
    ]

    # 4. Παράμετροι Συστήματος
    system_params = SystemParameters(
        prm_margin=1.15,            # 115% PRM
        co2_cap_tons=120000.0       # CO2 Limit για τις 336 ώρες
    )

    # 5. Δημιουργία ExpansionInput
    inputs = ExpansionInput(
        system_params=system_params,
        existing_fleet=existing_fleet,
        candidate_fleet=candidate_fleet,
        demand_profile=demand_profile,
        solar_cfs={"Sol1": solar_cf},
        wind_cfs={"WindNew": wind_cf}
    )

    print("⚙️ Εκτέλεση MILP Optimization (GLPK solver)...")
    
    try:
        results = build_and_solve_expansion(inputs, solver_name="glpk")
    except Exception as e:
        print(f"❌ Σφάλμα: {e}")
        return

    # 6. Εκτύπωση Αποτελεσμάτων
    print("\n" + "="*55)
    print("📊 ΑΠΟΤΕΛΕΣΜΑΤΑ EXPANSION OPTIMIZATION (MILP)")
    print("="*55)
    print(f"Solver Status:          {results.status}")
    print(f"Συνολικό Κόστος:        €{results.total_cost:,.2f}")
    print(f"Συνολικές Εκπομπές CO2:  {results.co2_emissions_tons:,.2f} Tons")
    print("-" * 55)
    print("🏗️  ΕΠΕΝΔΥΤΙΚΕΣ ΑΠΟΦΑΣΕΙΣ (NEW UNITS BUILT):")
    for unit, count in results.units_built.items():
        mw = results.new_capacity_mw[unit]
        print(f"  • {unit:<10}: {int(count)} μονάδες ({mw:,.0f} MW)")
    
    print("-" * 55)
    print("⚡ ΣΥΝΟΛΙΚΗ ΠΑΡΑΓΩΓΗ ΑΝΑ ΜΟΝΑΔΑ (MWh):")
    for gen, mwh in results.generation_mwh.items():
        print(f"  • {gen:<10}: {mwh:,.2f} MWh")
    print("="*55)

if __name__ == "__main__":
    run_test()

🚀 Προετοιμασία δεδομένων βασισμένων στον αρχικό κώδικα...
⚙️ Εκτέλεση MILP Optimization (GLPK solver)...

📊 ΑΠΟΤΕΛΕΣΜΑΤΑ EXPANSION OPTIMIZATION (MILP)
Solver Status:          ok
Συνολικό Κόστος:        €14,003,251.28
Συνολικές Εκπομπές CO2:  29,350.45 Tons
-------------------------------------------------------
🏗️  ΕΠΕΝΔΥΤΙΚΕΣ ΑΠΟΦΑΣΕΙΣ (NEW UNITS BUILT):
  • WindNew   : 32 μονάδες (3,200 MW)
  • CCNew     : 0 μονάδες (0 MW)
  • CTNew     : 7 μονάδες (700 MW)
-------------------------------------------------------
⚡ ΣΥΝΟΛΙΚΗ ΠΑΡΑΓΩΓΗ ΑΝΑ ΜΟΝΑΔΑ (MWh):
  • CC1       : 40,863.62 MWh
  • CC2       : 15,854.77 MWh
  • CT1       : 5,627.72 MWh
  • Nuc1      : 116,300.40 MWh
  • Sol1      : 26,585.14 MWh
  • WindNew   : 392,834.42 MWh
  • CCNew     : 0.00 MWh
  • CTNew     : 7,475.81 MWh


In [3]:
import numpy as np
from schemas import (
    ExistingGenerator,
    CandidateGenerator,
    SystemParameters,
    ExpansionInput
)
from CE_model import build_and_solve_expansion
from plots import plot_expansion_results

def run():
    print("🚀 Προετοιμασία δεδομένων...")

    # 1. Χρονοσειρά ζήτησης (24 ώρες * 14 ημέρες = 336 ώρες)
    hours = 336
    t = np.arange(hours)
    base_demand = 1800 + 700 * np.sin(2 * np.pi * t / 24) + np.random.normal(0, 30, hours)
    demand_profile = np.maximum(base_demand, 400).tolist()

    solar_cf = np.maximum(0, np.sin(2 * np.pi * (t - 6) / 24)).tolist()
    wind_cf = np.clip(0.40 + 0.20 * np.sin(2 * np.pi * t / 80) + np.random.normal(0, 0.05, hours), 0, 1).tolist()

    # 2. Υπάρχον Fleet
    existing_fleet = [
        ExistingGenerator(name="CC1", fuel_type="Natural Gas", capacity_mw=400.0, heat_rate=7.2, fuel_cost=4.5, vom_cost=3.0, co2_tons_per_mwh=0.35),
        ExistingGenerator(name="CC2", fuel_type="Natural Gas", capacity_mw=300.0, heat_rate=7.5, fuel_cost=4.5, vom_cost=3.5, co2_tons_per_mwh=0.40),
        ExistingGenerator(name="CT1", fuel_type="Natural Gas", capacity_mw=150.0, heat_rate=10.0, fuel_cost=4.5, vom_cost=5.0, co2_tons_per_mwh=0.75),
        ExistingGenerator(name="Nuc1", fuel_type="Nuclear", capacity_mw=600.0, heat_rate=10.4, fuel_cost=0.8, vom_cost=2.0, co2_tons_per_mwh=0.0),
        ExistingGenerator(name="Sol1", fuel_type="Solar", capacity_mw=250.0, heat_rate=0.0, fuel_cost=0.0, vom_cost=1.0, co2_tons_per_mwh=0.0, is_variable=True)
    ]

    # 3. Νέες Υποψήφιες Μονάδες (vb)
    candidate_fleet = [
        CandidateGenerator(name="WindNew", fuel_type="Wind", unit_capacity_mw=100.0, annual_capex_per_mw=65000.0, op_cost_per_mwh=2.0, co2_tons_per_mwh=0.0, is_variable=True, is_integer=True),
        CandidateGenerator(name="CCNew", fuel_type="Natural Gas", unit_capacity_mw=250.0, annual_capex_per_mw=85000.0, op_cost_per_mwh=35.0, co2_tons_per_mwh=0.30, is_variable=False, is_integer=True),
        CandidateGenerator(name="CTNew", fuel_type="Natural Gas", unit_capacity_mw=100.0, annual_capex_per_mw=45000.0, op_cost_per_mwh=65.0, co2_tons_per_mwh=0.60, is_variable=False, is_integer=True)
    ]

    # ---------------------------------------------------------
    # ΣΕΝΑΡΙΟ 1: Χωρίς CO2 Cap
    # ---------------------------------------------------------
    print("⚙️  [1/2] Επίλυση Σεναρίου ΧΩΡΙΣ CO2 Cap...")
    input_no_cap = ExpansionInput(
        system_params=SystemParameters(prm_margin=1.15, co2_cap_tons=None),
        existing_fleet=existing_fleet, candidate_fleet=candidate_fleet,
        demand_profile=demand_profile, solar_cfs={"Sol1": solar_cf}, wind_cfs={"WindNew": wind_cf}
    )
    results_no_cap = build_and_solve_expansion(input_no_cap)

    # ---------------------------------------------------------
    # ΣΕΝΑΡΙΟ 2: Με CO2 Cap
    # ---------------------------------------------------------
    co2_limit = 60000.0  # Όριο CO2 σε τόνους
    print(f"⚙️  [2/2] Επίλυση Σεναρίου ΜΕ CO2 Cap ({co2_limit:,.0f} Tons)...")
    input_cap = ExpansionInput(
        system_params=SystemParameters(prm_margin=1.15, co2_cap_tons=co2_limit),
        existing_fleet=existing_fleet, candidate_fleet=candidate_fleet,
        demand_profile=demand_profile, solar_cfs={"Sol1": solar_cf}, wind_cfs={"WindNew": wind_cf}
    )
    results_cap = build_and_solve_expansion(input_cap)

    # ---------------------------------------------------------
    # Προβολή Γραφημάτων
    # ---------------------------------------------------------
    print("📊 Δημιουργία και εμφάνιση διαγραμμάτων...")
    plot_expansion_results(results_no_cap, results_cap, demand_profile, co2_limit)

if __name__ == "__main__":
    run()

ImportError: cannot import name 'OptimizationResult' from 'schemas' (C:\Users\hsofi\OneDrive\Desktop\github\optergy-core\Capacity Expansion\schemas.py)